# 11.8 Case-study update-level zoom plots (reconstruct + re-anchor)

Fine-grained (5-min) companion to the 8h RIB panels. Reads the processed pickles written by
`scripts/11.7` (`{case}_anchors.pkl` = 8h RIB ground truth, `{case}_events.pkl` = 5-min `as_path`
update events) and replays them into a standing per-5-min timeline of announced prefixes and peers.

**Uniform reconstruct-and-reanchor.** At every 8h RIB anchor the per-`(collector, peer, prefix)` state
is reset to the RIB (kills update-stream drift / session-reset zombies); the 5-min updates are applied
within each bucket. Both lines come from the same state:
- **announced prefixes** (nb 4/6) = prefixes seen by >=95% collectors on a path where the case AS lies
  at or after the *rightmost* Tier-1 (the Tier-1's downstream cone). General rule: origin-only for ASes
  that announce their own space (BCE, IVOCS, SupplyNet), and also counts a transit case AS's customer
  prefixes (BelCloud, paths `... TIER1 44901 <customer>`),
- **peers** = distinct ASes adjacent to the case AS in any standing path (nb 3).

**Silent-feed backdating.** A route the *next* RIB shows gone but that never received an explicit
withdrawal (a session-reset zombie) gets a synthetic withdrawal at its peer's last observed activity,
so peer drops resolve to the real teardown time instead of snapping at the 8h re-anchor.

Data and rendering are separated: the replay caches each case's timeline to CSV, so cosmetic plot
edits (font, xlim, labels) re-run only the plot cell.

In [ ]:
import os
import sys
import json
import collections

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

sys.path.append("../scripts")
from case_studies import CASES

In [ ]:
REPO_ROOT = os.path.abspath("..")
with open(os.path.join(REPO_ROOT, "settings.json")) as fd:
    parameters = json.load(fd)
    for _k in ("DATA_DIR", "DATA_RAW_DIR", "IMAGE_DIR", "WORKING_DIR", "VISIBILITY_OUTPUT_DIR", "VISIBILITY_ANNOUNCED_OUTPUT_DIR"):
        if isinstance(parameters.get(_k), str) and not os.path.isabs(parameters[_k]):
            parameters[_k] = os.path.normpath(os.path.join(REPO_ROOT, parameters[_k]))

data_dir = parameters["DATA_DIR"]
start_date = parameters["START_DATE"]
end_date = parameters["END_DATE"]
image_dir = parameters["IMAGE_DIR"]
n_collectors = len(parameters["COLLECTORS"])

HV_MIN = int(np.ceil(0.95 * n_collectors))            # >=95% collector visibility (22/23), matches the paper
TIER1 = frozenset(t["asn"] for t in parameters["TIER1"])
EVENTS_DIR = f"{data_dir}/processed/updates_cases"    # {case}_anchors.pkl + {case}_events.pkl (from 11.7)
OUT_IMG = f"{image_dir}/case_study_zoom"
os.makedirs(OUT_IMG, exist_ok=True)

# PeeringDB limit series (orange line)
df_peeringdb = pd.read_pickle(
    f"{data_dir}/processed/peeringdb/prefix_limit_peeringdb_{start_date}_{end_date}.pkl"
)
print(f"{n_collectors} collectors -> HV_MIN={HV_MIN} | {len(TIER1)} Tier-1 ASes")

## Stateful replay (seed from RIB + apply updates)

In [ ]:
def adjacencies(as_path, asn):
    """Set of ASes directly adjacent to `asn` anywhere in the path (skip prepends).
    Matches the notebook-3 peer definition (all AS-path adjacencies, not only origin-side)."""
    a = as_path.split()
    s = str(asn)
    out = set()
    for i, h in enumerate(a):
        if h != s:
            continue
        if i > 0 and a[i - 1] != s:
            out.add(int(a[i - 1]))
        if i + 1 < len(a) and a[i + 1] != s:
            out.add(int(a[i + 1]))
    return out


def counts_for(as_path, asn):
    """True if this path makes `prefix` an *announced prefix* of the case AS, per nb4/6.

    nb4 rule: find the Tier-1 closest to the origin (rightmost in the path) and attribute the
    prefix to every AS from that Tier-1 down to the origin (the Tier-1's downstream cone).
    So the case AS counts iff it appears at or after the rightmost Tier-1. This is the GENERAL
    definition: it reduces to "case AS is the origin" for ASes that announce their own space
    (BCE, IVOCS, SupplyNet), and also captures a transit case AS's customer prefixes -- e.g.
    BelCloud, where the counted prefixes ride paths `... TIER1 44901 <customer>`."""
    a = as_path.split()
    if "{" in as_path or len(a) < 2:            # skip AS_SET notation, matches nb4
        return False
    last_t1 = -1
    for i in range(len(a) - 1, -1, -1):
        try:
            if int(a[i]) in TIER1:
                last_t1 = i
                break
        except ValueError:
            return False
    return last_t1 >= 0 and str(asn) in a[last_t1:]


def _snapshot(state, asn):
    """(peer count, announced-prefix count) from the current standing state."""
    peers = set()
    pfx_coll = collections.defaultdict(set)
    for (coll, peer), d in state.items():
        for pfx, path in d.items():
            peers |= adjacencies(path, asn)
            if counts_for(path, asn):
                pfx_coll[pfx].add(coll)
    pfx_hv = sum(1 for cs in pfx_coll.values() if len(cs) >= HV_MIN)
    return len(peers), pfx_hv


def replay(name, case, interval=300, backdate=True):
    """Uniform reconstruct-and-reanchor replay for BOTH lines (see the header cell).

    `backdate` synthesises a withdrawal for session-reset zombies -- routes the next RIB shows
    gone but that never received an explicit withdrawal -- at the peer's last observed activity
    for the case AS, narrowing peer drops from the 8h re-anchor to the real teardown time.

    Memory note: the events pickle for a big transit case AS (e.g. BelCloud, ~4 GB) is far too
    large to materialise as a list of namedtuples. We keep the columns as numpy arrays and use
    `searchsorted` to slice one 8h bucket at a time, so only a single bucket is ever expanded.
    """
    asn = case["asn"]
    anch = pd.read_pickle(f"{EVENTS_DIR}/{name}_anchors.pkl")
    ev = (pd.read_pickle(f"{EVENTS_DIR}/{name}_events.pkl")
          .sort_values("ts", kind="stable").reset_index(drop=True))
    anchors = sorted(anch["anchor_ts"].unique())
    by_anchor = {a: g for a, g in anch.groupby("anchor_ts")}
    rib_keys = {a: set(zip(g.collector, g.peer_asn, g.prefix)) for a, g in anch.groupby("anchor_ts")}

    # column arrays (avoid one giant list of namedtuples); ts is sorted -> searchsorted per bucket
    e_ts = ev["ts"].to_numpy()
    e_coll = ev["collector"].to_numpy()
    e_peer = ev["peer_asn"].to_numpy()
    e_type = ev["type"].to_numpy()
    e_pfx = ev["prefix"].to_numpy()
    e_path = ev["as_path"].to_numpy()
    del ev

    def seed(a0):
        st = collections.defaultdict(dict)
        for r in by_anchor[a0].itertuples(index=False):
            st[(r.collector, r.peer_asn)][r.prefix] = r.as_path
        return st

    out = []
    for bi, a0 in enumerate(anchors):
        a1 = anchors[bi + 1] if bi + 1 < len(anchors) else a0 + 8 * 3600
        lo = int(np.searchsorted(e_ts, a0, "left"))
        hi = int(np.searchsorted(e_ts, a1, "left"))

        # synthetic withdrawals for zombies (present at bucket end, gone in next RIB, no W seen)
        synth = []
        if backdate and (bi + 1) < len(anchors):
            state = seed(a0)
            last_act = {}
            for j in range(lo, hi):
                key = (e_coll[j], e_peer[j])
                last_act[key] = e_ts[j]
                if e_type[j] == "A":
                    state[key][e_pfx[j]] = e_path[j]
                else:
                    state[key].pop(e_pfx[j], None)
            nxt = rib_keys[anchors[bi + 1]]
            for (coll, peer), d in state.items():
                for pfx in d:
                    if (coll, peer, pfx) not in nxt:
                        synth.append((last_act.get((coll, peer), a0), coll, peer, "W", pfx, None))

        # expand only this bucket's real events, merge with synthetic, replay
        merged = [(e_ts[j], e_coll[j], e_peer[j], e_type[j], e_pfx[j], e_path[j])
                  for j in range(lo, hi)] + synth
        merged.sort(key=lambda x: x[0])
        state = seed(a0)
        ei, nb = 0, len(merged)
        for tk in range(int(a0), int(a1), interval):
            while ei < nb and merged[ei][0] <= tk:
                _, c, p, t, pfx, path = merged[ei]
                key = (c, p)
                if t == "A":
                    state[key][pfx] = path
                else:
                    state[key].pop(pfx, None)
                ei += 1
            npeers, pfx_hv = _snapshot(state, asn)
            out.append((tk, npeers, pfx_hv))

    df = pd.DataFrame(out, columns=["ts", "peers", "pfx_hv"])
    df["time"] = pd.to_datetime(df["ts"], unit="s", utc=True)
    return df, anchors

In [ ]:
# Compute each case's 5-min timeline once and cache it to CSV, so cosmetic plot edits
# (font, xlim, labels) re-run only the plot cell below -- never this heavier replay.
TIMELINES = {}
CASE_ANCHORS = {}
REPLAY_FORCE = False   # True = re-run heavy replay; else reuse cached *_timeline.csv
for _name, _case in CASES.items():
    _csv = f"{EVENTS_DIR}/{_name}_timeline.csv"
    if os.path.exists(_csv) and not REPLAY_FORCE:
        TIMELINES[_name] = pd.read_csv(_csv); CASE_ANCHORS[_name] = None
        print(f"[{_name}] cached timeline reused ({len(TIMELINES[_name])} samples) -- set REPLAY_FORCE=True to recompute")
        continue
    if not (os.path.exists(f"{EVENTS_DIR}/{_name}_anchors.pkl")
            and os.path.exists(f"{EVENTS_DIR}/{_name}_events.pkl")):
        print(f"[{_name}] skip (run scripts/11.7 first)")
        continue
    _df, _anchors = replay(_name, _case)
    _df.to_csv(f"{EVENTS_DIR}/{_name}_timeline.csv", index=False)
    TIMELINES[_name] = _df
    CASE_ANCHORS[_name] = _anchors
    print(f"[{_name}] pfx {_df.pfx_hv.min()}..{_df.pfx_hv.max()} | peers {_df.peers.min()}..{_df.peers.max()} "
          f"| {len(_df)} 5-min samples -> {_name}_timeline.csv")

## Plot (11.5 style: green prefixes, orange limit, purple peers)

In [ ]:
# ---- cosmetic knobs (edit freely; re-run this cell + the run cell only) -------------------
FONT_SIZE = 24        # match 11.5 (the RIB panels) so the grid columns share type size
FIGSIZE = (16, 6)     # match 11.5
LINE_WIDTH = 3        # match 11.5
XTICK_ROT = 45        # match 11.5 (plt.xticks(rotation=45, ha="center"))
COLOR_PFX = "tab:green"
COLOR_LIMIT = "tab:orange"
COLOR_PEERS = "tab:purple"
COLOR_CROSS = "tab:red"
LABEL_PFX = "Announced Prefixes"
LABEL_LIMIT = "Max-Prefix Limit"
LABEL_PEERS = "Number of Peers"
SHOW_ANCHORS = False   # off: the 1h/4h x-grid below now provides structure
SHOW_DETECT = False    # red dotted 8h RIB-detection line: adds little value, kept off (2026-08 review)
# NOTE: no per-point markers -- the dense 5-min series and its natural noise already signal the
# finer temporal resolution relative to the RIB panels.

# per-case zoom window: ~10h either side of the real 5-min crossing (tight, so the fast events
# are readable); BelCloud runs longer on the right to capture the limit raise + peer restoration.
# Keep these identical to case_study_events[*].zoom_{start,end} in 11.5 so the axvspan lines up.
ZOOM = {
    "AS44901_BelCloud_v6":  ("2025-01-15 04:00", "2025-01-16 12:00"),
    "AS25273_BCE_v4":       ("2025-09-04 19:00", "2025-09-05 15:00"),
    "AS52920_IVOCS_v4":     ("2025-08-11 06:00", "2025-08-12 02:00"),
    "AS52603_SupplyNet_v6": ("2025-09-01 08:00", "2025-09-02 04:00"),
}
# the 8h RIB snapshot at which the crossing is *detected* (only drawn if SHOW_DETECT).
DETECT = {
    "AS44901_BelCloud_v6":  "2025-01-15 16:00",
    "AS25273_BCE_v4":       "2025-09-05 08:00",
    "AS52920_IVOCS_v4":     "2025-08-12 00:00",
    "AS52603_SupplyNet_v6": "2025-09-02 00:00",
}
# left (prefixes/limit), right (peers) y-limits -- HARDCODED to match the RIB panels in 11.5
# (case_study_events min/max_peers, and max(prefix,limit)*1.1) so each RIB | zoom pair shares axes.
YLIM = {
    "AS44901_BelCloud_v6":  ((0, 330),  (450, 950)),
    "AS25273_BCE_v4":       ((0, 36.3), (10, 20)),
    "AS52920_IVOCS_v4":     ((0, 28.6), (0, 30)),
    "AS52603_SupplyNet_v6": ((0, 15),   (0, 25)),
}
# left, right y-TICKS -- FORCED to the RIB panels' actual rendered ticks (read from the 11.5
# case_study PDFs) so each RIB | zoom pair shares identical gridlines, not just the same range.
YTICKS = {
    "AS44901_BelCloud_v6":  ([0, 40, 80, 120, 160, 200, 240, 280, 320], [500, 600, 700, 800, 900]),
    "AS25273_BCE_v4":       ([0, 4, 8, 12, 16, 20, 24, 28, 32, 36],     [10, 12, 14, 16, 18, 20]),
    "AS52920_IVOCS_v4":     ([0, 3, 6, 9, 12, 15, 18, 21, 24, 27],      [0, 5, 10, 15, 20, 25, 30]),
    "AS52603_SupplyNet_v6": ([0, 2, 4, 6, 8, 10, 12, 14],              [0, 5, 10, 15, 20, 25]),
}
# ------------------------------------------------------------------------------------------


def limit_series(asn, ipv, t0, t1):
    """PeeringDB limit (ipv) as a step spanning the FULL [t0, t1] window. The limit is sampled
    at 8h/daily boundaries, so we clamp: carry the value active at t0 to the left edge and hold
    the last in-window value out to t1, otherwise the orange line stops short of the axis."""
    row = df_peeringdb[df_peeringdb["asn"] == asn]
    if row.empty:
        return [], []
    row = row.iloc[0]
    vals = row[f"limits_ipv{ipv}"]
    if vals is None:
        return [], []
    times = pd.to_datetime(pd.Series(row["dates"]), utc=True)
    s = pd.Series(list(vals), index=times).sort_index()
    if s.empty:
        return [], []
    before = s[s.index <= t0]
    v0 = before.iloc[-1] if len(before) else s.iloc[0]
    inwin = s[(s.index > t0) & (s.index < t1)]
    xs = [t0] + list(inwin.index) + [t1]
    ys = [v0] + list(inwin.values) + [inwin.iloc[-1] if len(inwin) else v0]
    return xs, ys


def plot_case_zoom(name, case, df, anchors=None, save=True):
    asn, ipv = case["asn"], case["ipv"]
    df = df.copy()
    df["time"] = pd.to_datetime(df["time"], utc=True)
    win = ZOOM.get(name)
    t0 = pd.Timestamp(win[0], tz="UTC") if win else df["time"].iloc[0]
    t1 = pd.Timestamp(win[1], tz="UTC") if win else df["time"].iloc[-1]
    df = df[(df["time"] >= t0) & (df["time"] <= t1)]

    plt.rcParams["font.size"] = FONT_SIZE
    fig, ax1 = plt.subplots(figsize=FIGSIZE)

    # left axis: announced prefixes (>=95% visibility, nb 4/6) + PeeringDB limit -- no markers
    ax1.step(df["time"], df["pfx_hv"], where="mid", color=COLOR_PFX, lw=LINE_WIDTH, label=LABEL_PFX)
    lt, lv = limit_series(asn, ipv, t0, t1)
    if lt:
        ax1.step(lt, lv, where="mid", color=COLOR_LIMIT, lw=LINE_WIDTH, label=LABEL_LIMIT)
    ax1.set_ylabel("Number of Prefixes / Limit")
    if name in YLIM:
        ax1.set_ylim(*YLIM[name][0])
    else:
        ax1.set_ylim(0, None)
    if name in YTICKS:
        ax1.set_yticks(YTICKS[name][0])          # forced to match the RIB panel
    else:
        ax1.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

    # right axis: peer count -- no markers, y-range + y-ticks matched to the RIB panel
    ax2 = ax1.twinx()
    ax2.step(df["time"], df["peers"], where="mid", color=COLOR_PEERS, lw=LINE_WIDTH, label=LABEL_PEERS)
    ax2.set_ylabel("Number of Peers")
    if name in YLIM:
        ax2.set_ylim(*YLIM[name][1])
    if name in YTICKS:
        ax2.set_yticks(YTICKS[name][1])          # forced to match the RIB panel
    else:
        ax2.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

    # faint 8h re-anchor gridlines; the red dotted RIB-detection line is off (SHOW_DETECT)
    if SHOW_DETECT:
        det = DETECT.get(name)
        if det:
            ax1.axvline(pd.Timestamp(det, tz="UTC"), color=COLOR_CROSS, lw=2, ls=":", alpha=0.9)
    if SHOW_ANCHORS:
        a = t0.normalize()
        while a <= t1:
            if a >= t0:
                ax1.axvline(a, color="grey", lw=0.6, alpha=0.25)
            a += pd.Timedelta(hours=8)

    # x-axis: sparse ticks at 45 deg, matching 11.5 (avoids the crowded 2-hourly labels)
    ax1.set_xlim(t0, t1)
    ax1.set_axisbelow(True)
    # labels only every 4h; an unlabeled tick + faint gridline every 1h -> easier to read
    ax1.xaxis.set_major_locator(mdates.HourLocator(byhour=range(0, 24, 4)))
    # ax1.xaxis.set_minor_locator(mdates.HourLocator(byhour=range(0, 24, 1)))
    ax1.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    for lab in ax1.get_xticklabels():
        lab.set_rotation(XTICK_ROT)
        lab.set_ha("center")
    ax1.grid(axis="y", alpha=0.3)
    # ax1.grid(axis="x", which="minor", color="grey", lw=0.4, alpha=0.20)  # 1h gridlines
    ax1.grid(axis="x", which="major", color="grey", lw=0.6, alpha=0.30)  # 4h gridlines

    if save:
        f = f"{OUT_IMG}/{name}_zoom"
        fig.savefig(f"{f}.pdf", bbox_inches="tight", dpi=300)
        fig.savefig(f"{f}.png", bbox_inches="tight", dpi=300)
        # mirror to tmp_plot so it renders in VS Code
        tmp = os.path.join(REPO_ROOT, "tmp_plot")
        os.makedirs(tmp, exist_ok=True)
        fig.savefig(f"{tmp}/{name}_zoom.png", bbox_inches="tight", dpi=130)
    plt.show()
    return fig


## Run (cases whose seed + events pickles exist)

In [ ]:
# Plot every case with a cached timeline. Cosmetic-only edits: tweak the knobs in the plot
# cell and re-run this cell (it reads the cached CSV if the in-memory timeline is absent).
for name, case in CASES.items():
    csv = f"{EVENTS_DIR}/{name}_timeline.csv"
    if name in TIMELINES:
        df, anchors = TIMELINES[name], CASE_ANCHORS.get(name)
    elif os.path.exists(csv):
        df, anchors = pd.read_csv(csv), None
    else:
        print(f"[{name}] skip (no timeline; run the compute cell / scripts/11.7)")
        continue
    print(f"[{name}] peers {df.peers.min()}..{df.peers.max()} | pfx {df.pfx_hv.min()}..{df.pfx_hv.max()}")
    plot_case_zoom(name, case, df, anchors)

## Shared legend (single row, like 11.5)

In [ ]:
plt.figure(figsize=(15, 1))
plt.axis("off")
plt.step([], [], color="tab:green", lw=8, label="Announced Prefixes")
plt.step([], [], color="tab:orange", lw=8, label="Max-Prefix Limit")
plt.step([], [], color="tab:purple", lw=8, label="Number of Peers")
plt.xlim(1, 2)
plt.legend(loc="upper right", ncol=3, frameon=False, fontsize=20)
plt.savefig(f"{OUT_IMG}/legend_single_row.pdf", bbox_inches="tight", dpi=300)
plt.savefig(f"{OUT_IMG}/legend_single_row.png", bbox_inches="tight", dpi=300)
plt.show()
